In [1]:
import os
import sys

import math
import time
import datetime
import numpy as np
from numpy.lib.stride_tricks import sliding_window_view
import torch
from torch.utils.data import Dataset
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torch.optim.lr_scheduler import CosineAnnealingLR
from matcho_3d import Unet3D
# from YourDataset import YourDataset  # Import your custom dataset here
from tqdm import tqdm
from torch.cuda.amp import autocast, GradScaler
from torchinfo import summary
import torchprofile

import pickle

torch.manual_seed(23)

scaler = GradScaler()

DTYPE = torch.float32
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams["figure.dpi"] = 200
plt.rcParams["font.family"] = "serif"

import scipy.stats as stats

/oscar/home/voommen/apps/torch_env/lib64/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cuda


In [2]:
# Define your custom loss function here
class CustomLoss(nn.Module):
    def __init__(self, Par):
        super(CustomLoss, self).__init__()
        self.Par = Par

    def forward(self, y_pred, y_true):
        # Implement your custom loss calculation here
        # loss = torch.mean((y_pred - y_true) ** 2)  # Example: Mean Squared Error
        y_true = (y_true - self.Par["out_shift"])/self.Par["out_scale"]
        y_pred = (y_pred - self.Par["out_shift"])/self.Par["out_scale"]
        loss = torch.norm(y_true-y_pred, p=2)/torch.norm(y_true, p=2)
        return loss

class YourDataset(Dataset):
    def __init__(self, x, t, y, transform=None):
        self.x = x
        self.t = t
        self.y = y
        self.transform = transform

    def __len__(self):
        return len(self.x)

    def __getitem__(self, idx):
        x_sample = self.x[idx]
        t_sample = self.t[idx]
        y_sample = self.y[idx]

        if self.transform:
            x_sample, t_sample, y_sample = self.transform(x_sample, t_sample, y_sample)

        return x_sample, t_sample, y_sample


# def preprocess(traj, Par):
#     x = sliding_window_view(traj[:,:-(Par['lf']),:,:,:,:], window_shape=Par['lb'], axis=1 ).transpose(0,1,6,2,3,4,5).reshape(-1,Par['lb'], Par['nf'], Par['nth'], Par['nr'], Par['nx'])
#     y = sliding_window_view(traj[:,Par['lb']:,:,:,:,:], window_shape=Par['lf'], axis=1 ).transpose(0,1,6,2,3,4,5).reshape(-1,Par['lf'], Par['nf'], Par['nth'], Par['nr'], Par['nx'])
#     t = np.linspace(0,1,Par['lf']).reshape(-1,1)

#     nt = y.shape[1]
#     n_samples = y.shape[0]

#     t = np.tile(t, [n_samples,1]).reshape(-1,)                                                     #[_*nt, ]
#     x = np.repeat(x,nt, axis=0)                                   #[_*nt, 1, 64, 64]
#     y = y.reshape(y.shape[0]*y.shape[1],1,y.shape[2],y.shape[3])  #[_*nt, 64, 64]


#     print('x: ', x.shape)
#     print('y: ', y.shape)
#     print('t: ', t.shape)
#     print()
#     return x,y,t

def preprocess(traj, Par):
    nsamples = traj.shape[0]
    nt = traj.shape[1]
    temp = nt - Par['lb'] - Par['lf'] + 1
    x_idx = np.arange(temp).reshape(-1,1)
    x_idx = np.tile(x_idx, (1, Par['lf'])).reshape(-1,)

    t_idx = np.arange(Par['lf']).reshape(1,-1)
    t_idx = np.tile(t_idx, (temp,1)).reshape(-1,)

    y_idx = np.arange(nt)
    y_idx = sliding_window_view(y_idx[Par['lb']:], window_shape=Par['lf']).reshape(-1,)

    print(f"x_idx: {x_idx.shape}")
    print(f"t_idx: {t_idx.shape}")
    print(f"y_idx: {y_idx.shape}")

    return x_idx, t_idx, y_idx


def combined_scheduler(optimizer, total_epochs, warmup_epochs, last_epoch=-1):
    def lr_lambda(epoch):
        if epoch < warmup_epochs:
            return float(epoch + 1) / warmup_epochs
        else:
            return 0.5 * (1 + math.cos(math.pi * (epoch - warmup_epochs) / (total_epochs - warmup_epochs)))

    return LambdaLR(optimizer, lr_lambda, last_epoch)

In [3]:
# Load your data into NumPy arrays (x_train, t_train, y_train, x_val, t_val, y_val, x_test, t_test, y_test)
#########################
debug = False

res = 128
begin_time = time.time()
if debug:
    traj = np.load(f"../data/velocity_sample.npy")[:,:,::2, ::2, ::2] #[nt, nf, nth, nr, nx]
else:
    traj = np.load("../data/velocity_vec_3d.npy")[:,:,::2, ::2, ::2]

traj = np.expand_dims(traj, axis=0) #[1, nt, nf, nth, nr, nx]
print(f"Data Loading Time: {time.time() - begin_time:.1f}s")

print(f"traj: {traj.shape}")

idx1 = int(0.8 * traj.shape[1])
idx2 = int(0.9 * traj.shape[1])

traj_train = traj[:, :idx1]
traj_val   = traj[:, idx1:idx2]
traj_test  = traj[:, idx2:]


Par = {}
# Par['nt'] = 100 
Par['nf'] = traj_train.shape[2]
Par['nth'] = traj_train.shape[3]
Par['nr'] = traj_train.shape[4]
Par['nx'] = traj_train.shape[5]
Par['d_emb'] = 128

Par['lb'] = 1
Par['lf'] = 10
Par['channels'] = Par['nf']
# Par['temp'] = Par['nt'] - Par['lb'] - Par['lf'] + 2

Par['num_epochs'] = 50

time_cond = np.linspace(0, 1, Par['lf'])

begin_time = time.time()
print('\nTrain Dataset')
x_idx_train, t_idx_train, y_idx_train = preprocess(traj_train, Par)
print('\nValidation Dataset')
x_idx_val, t_idx_val, y_idx_val  = preprocess(traj_val, Par)
print('\nTest Dataset')
x_idx_test, t_idx_test, y_idx_test  = preprocess(traj_test, Par)
print(f"Data Preprocess Time: {time.time() - begin_time:.1f}s")

# sys.exit()

t_min = np.min(time_cond)
t_max = np.max(time_cond)

MEAN = np.load("../data/MEAN_vel.npy").reshape(1,-1,1,1,1)
STD  = np.load("../data/STD_vel.npy").reshape(1,-1,1,1,1)
MIN  = np.load("../data/MIN_vel.npy").reshape(1,-1,1,1,1)
MAX  = np.load("../data/MAX_vel.npy").reshape(1,-1,1,1,1)

Par['inp_shift'] = torch.tensor(MEAN, dtype=DTYPE, device=device)
Par['inp_scale'] = torch.tensor(STD, dtype=DTYPE, device=device)
Par['out_shift'] = torch.tensor(MEAN, dtype=DTYPE, device=device)
Par['out_scale'] = torch.tensor(STD, dtype=DTYPE, device=device)
Par['t_shift']   = t_min
Par['t_scale']   = t_max - t_min

with open('Par.pkl', 'wb') as f:
    pickle.dump(Par, f)

# sys.exit()
#########################

# Create custom datasets
traj_train_tensor = torch.tensor(traj_train, dtype=DTYPE)
traj_val_tensor = torch.tensor(traj_val, dtype=DTYPE)
traj_test_tensor = torch.tensor(traj_test, dtype=DTYPE)
time_cond_tensor = torch.tensor(time_cond, dtype=DTYPE)


train_dataset = YourDataset(x_idx_train, t_idx_train, y_idx_train)
val_dataset = YourDataset(x_idx_val, t_idx_val, y_idx_val)
test_dataset = YourDataset(x_idx_test, t_idx_test, y_idx_test)

# Define data loaders
train_batch_size = 10
val_batch_size   = 10
test_batch_size  = 10
train_loader = DataLoader(train_dataset, batch_size=train_batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=val_batch_size)
test_loader = DataLoader(test_dataset, batch_size=test_batch_size)

Data Loading Time: 4.0s
traj: (1, 1000, 3, 32, 32, 128)

Train Dataset
x_idx: (7900,)
t_idx: (7900,)
y_idx: (7900,)

Validation Dataset
x_idx: (900,)
t_idx: (900,)
y_idx: (900,)

Test Dataset
x_idx: (900,)
t_idx: (900,)
y_idx: (900,)
Data Preprocess Time: 0.0s


In [6]:
model = Unet3D(dim=16, Par=Par, dim_mults=(1, 2, 4, 8), channels=Par['channels']).to(device).to(torch.float32)

path_model = 'models/best_model.pt'
model.load_state_dict(torch.load(path_model))

print(summary(model, input_size=((1, 3, 32, 32, 128), (1,)) ) )

# Adjust the dimensions as per your model's input size
dummy_x = traj_train_tensor[0,[0]].to(device)
dummy_t = time_cond_tensor[0:1].to(device)
dummy_input = (dummy_x, dummy_t)

# Profile the model
flops = torchprofile.profile_macs(model, dummy_input)
print(f"FLOPs: {flops:.2e}")

# Define loss function and optimizer
criterion = CustomLoss(Par)

Layer (type:depth-idx)                                  Output Shape              Param #
Unet3D                                                  [1, 3, 32, 32, 128]       --
├─RelativePositionBias: 1-1                             [8, 32, 32]               --
│    └─Embedding: 2-1                                   [32, 32, 8]               256
├─Conv3d: 1-2                                           [1, 16, 32, 32, 128]      2,368
├─Residual: 1-3                                         [1, 16, 32, 32, 128]      --
│    └─PreNorm: 2-2                                     [1, 16, 32, 32, 128]      --
│    │    └─LayerNorm: 3-1                              [1, 16, 32, 32, 128]      16
│    │    └─EinopsToAndFrom: 3-2                        [1, 16, 32, 32, 128]      16,400
├─Sequential: 1-4                                       [1, 64]                   --
│    └─SinusoidalPosEmb: 2-3                            [1, 16]                   --
│    └─Linear: 2-4                                  

/oscar/home/voommen/apps/torch_env/lib64/python3.9/site-packages/torchprofile/profile.py:22: UserWarning: No handlers found: "aten::scalarimplicit". Skipped.
  warnings.warn('No handlers found: "{}". Skipped.'.format(
/oscar/home/voommen/apps/torch_env/lib64/python3.9/site-packages/torchprofile/profile.py:22: UserWarning: No handlers found: "aten::arange". Skipped.
  warnings.warn('No handlers found: "{}". Skipped.'.format(
/oscar/home/voommen/apps/torch_env/lib64/python3.9/site-packages/torchprofile/profile.py:22: UserWarning: No handlers found: "aten::reshape". Skipped.
  warnings.warn('No handlers found: "{}". Skipped.'.format(
/oscar/home/voommen/apps/torch_env/lib64/python3.9/site-packages/torchprofile/profile.py:22: UserWarning: No handlers found: "aten::neg". Skipped.
  warnings.warn('No handlers found: "{}". Skipped.'.format(
/oscar/home/voommen/apps/torch_env/lib64/python3.9/site-packages/torchprofile/profile.py:22: UserWarning: No handlers found: "aten::abs". Skipped.
  warni

# Sanity Check

In [7]:
y_true_ls = []
y_pred_ls = []

model.eval()
train_loss = 0.0
with torch.no_grad():
    for x_idx, t_idx, y_idx in train_loader:
        x = traj_train_tensor[0, x_idx]
        t = time_cond_tensor[t_idx]
        y_true = traj_train_tensor[0, y_idx]
        with autocast():
            y_pred = model(x.to(device), t.to(device))
            loss   = criterion(y_pred, y_true.to(device))
        train_loss += loss.item()
        y_true_ls.append(y_true.detach().cpu().numpy())
        y_pred_ls.append(y_pred.detach().cpu().numpy())

train_loss /= len(train_loader)
print(f"Train Loss: {train_loss:.4e}")

TRAIN_TRUE = np.concatenate(y_true_ls, axis=0).reshape(-1, Par['lf'], Par['nf'], Par['nth'], Par['nr'], Par['nx']).astype(np.float32)
TRAIN_PRED = np.concatenate(y_pred_ls, axis=0).reshape(-1, Par['lf'], Par['nf'], Par['nth'], Par['nr'], Par['nx']).astype(np.float32)

print(f"TRAIN_TRUE: {TRAIN_TRUE.shape}, DTYPE: {TRAIN_TRUE.dtype}")
print(f"TRAIN_PRED: {TRAIN_PRED.shape}, DTYPE: {TRAIN_PRED.dtype}")



y_true_ls = []
y_pred_ls = []

model.eval()
val_loss = 0.0
with torch.no_grad():
    for x_idx, t_idx, y_idx in val_loader:
        x = traj_val_tensor[0, x_idx]
        t = time_cond_tensor[t_idx]
        y_true = traj_val_tensor[0, y_idx]
        with autocast():
            y_pred = model(x.to(device), t.to(device))
            loss   = criterion(y_pred, y_true.to(device))
        val_loss += loss.item()
        y_true_ls.append(y_true.detach().cpu().numpy())
        y_pred_ls.append(y_pred.detach().cpu().numpy())

val_loss /= len(val_loader)
print(f"Val Loss: {val_loss:.4e}")

VAL_TRUE = np.concatenate(y_true_ls, axis=0).reshape(-1, Par['lf'], Par['nf'], Par['nth'], Par['nr'], Par['nx']).astype(np.float32)
VAL_PRED = np.concatenate(y_pred_ls, axis=0).reshape(-1, Par['lf'], Par['nf'], Par['nth'], Par['nr'], Par['nx']).astype(np.float32)

print(f"VAL_TRUE: {VAL_TRUE.shape}, DTYPE: {VAL_TRUE.dtype}")
print(f"VAL_PRED: {VAL_PRED.shape}, DTYPE: {VAL_PRED.dtype}")



y_true_ls = []
y_pred_ls = []

model.eval()
test_loss = 0.0
with torch.no_grad():
    for x_idx, t_idx, y_idx in test_loader:
        x = traj_test_tensor[0, x_idx]
        t = time_cond_tensor[t_idx]
        y_true = traj_test_tensor[0, y_idx]
        with autocast():
            y_pred = model(x.to(device), t.to(device))
            loss   = criterion(y_pred, y_true.to(device))
        test_loss += loss.item()
        y_true_ls.append(y_true.detach().cpu().numpy())
        y_pred_ls.append(y_pred.detach().cpu().numpy())

test_loss /= len(test_loader)
print(f"Test Loss: {test_loss:.4e}")

TEST_TRUE = np.concatenate(y_true_ls, axis=0).reshape(-1, Par['lf'], Par['nf'], Par['nth'], Par['nr'], Par['nx']).astype(np.float32)
TEST_PRED = np.concatenate(y_pred_ls, axis=0).reshape(-1, Par['lf'], Par['nf'], Par['nth'], Par['nr'], Par['nx']).astype(np.float32)

print(f"TEST_TRUE: {TEST_TRUE.shape}, DTYPE: {TEST_TRUE.dtype}")
print(f"TEST_PRED: {TEST_PRED.shape}, DTYPE: {TEST_PRED.dtype}")

Train Loss: 6.5081e-01
TRAIN_TRUE: (790, 10, 3, 32, 32, 128), DTYPE: float32
TRAIN_PRED: (790, 10, 3, 32, 32, 128), DTYPE: float32
Val Loss: 6.7842e-01
VAL_TRUE: (90, 10, 3, 32, 32, 128), DTYPE: float32
VAL_PRED: (90, 10, 3, 32, 32, 128), DTYPE: float32
Test Loss: 6.8016e-01
TEST_TRUE: (90, 10, 3, 32, 32, 128), DTYPE: float32
TEST_PRED: (90, 10, 3, 32, 32, 128), DTYPE: float32


In [22]:
np.save("TRAIN_TRUE.npy", TRAIN_TRUE)
np.save("TRAIN_PRED.npy", TRAIN_PRED)

np.save("VAL_TRUE.npy", VAL_TRUE)
np.save("VAL_PRED.npy", VAL_PRED)

np.save("TEST_TRUE.npy", TEST_TRUE)
np.save("TEST_PRED.npy", TEST_PRED)

In [7]:
sample1 = TRAIN_TRUE[1]
sample2 = TRAIN_TRUE[51]

err = np.abs(sample1 - sample2)
print(f"max err: {np.max(err)}")
print(f"min err: {np.min(err)}")

max err: 0.15937381982803345
min err: 0.0
